In [ ]:
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from dataloader import get_dataloaders
import torch
import torch.nn as nn
import torch.optim as optim
import os


In [ ]:
train_loader, val_loader, test_loader, label_to_idx = get_dataloaders(data_dir="data", batch_size=32)

print(f"class mapping: {label_to_idx}")
print(f"train batches: {len(train_loader)},  val batches: {len(val_loader)}, test batches: {len(test_loader)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_classes = len(label_to_idx)


class BeerCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 64 * 64, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = BeerCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def evaluate_model(model, train_loader, val_loader, criterion, device, num_classes):
    model.eval()
    results = {}

    with torch.no_grad():
        for split, loader in [("train", train_loader), ("val", val_loader)]:
            all_labels = []
            all_preds = []
            all_probs = []
            total_loss = 0.0

            for images, labels in loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                total_loss += loss.item() * images.size(0)
                probs = torch.softmax(outputs, dim=1)
                _, predicted = outputs.max(1)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

            all_labels = np.array(all_labels)
            all_preds = np.array(all_preds)
            all_probs = np.array(all_probs)

            results[f"{split}_loss"] = total_loss / len(all_labels)
            results[f"{split}_accuracy"] = accuracy_score(all_labels, all_preds)
            results[f"{split}_f1_score"] = f1_score(all_labels, all_preds, average="weighted")
            try:
                results[f"{split}_roc_auc"] = roc_auc_score(
                    all_labels, all_probs, multi_class="ovr", average="weighted"
                )
            except ValueError:
                results[f"{split}_roc_auc"] = float("nan")
            results[f"{split}_confusion_matrix"] = confusion_matrix(
                all_labels, all_preds, labels=list(range(num_classes))
            )

    return results

In [ ]:
num_epochs = 20
training_records = []


for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

        pbar.set_postfix(train_loss=train_loss / train_total)

    results_record = evaluate_model(
        model, train_loader, val_loader, criterion, device, num_classes
    )
    results_record["epoch"] = epoch + 1
    training_records.append(results_record)
    print(f"val loss: {results_record['val_loss']:.4f}")


In [ ]:
run_name = input("Enter run name: ")
output_dir = os.path.join("outputs", run_name)
os.makedirs(output_dir, exist_ok=True)

# Save model
torch.save(model.state_dict(), os.path.join(output_dir, "model.pt"))

# Save metrics (exclude confusion matrices from CSV)
metrics_df = pd.DataFrame(training_records)
metrics_df.to_csv(os.path.join(output_dir, "metrics.csv"), index=False)

print(f"Saved to {output_dir}/")
display(metrics_df)